In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shahddaymann/libraries-docs/pandas_complete_guide.pdf
/kaggle/input/datasets/shahddaymann/libraries-docs/scikit-learn-docs.pdf
/kaggle/input/datasets/shahddaymann/libraries-docs/NumPy_Complete_Guide.pdf
/kaggle/input/datasets/shahddaymann/libraries-docs/numpy-ref.pdf


In [2]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 8.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storag

In [3]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re
import json
from collections import Counter

/tmp/ipykernel_58/1788431493.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


# **Document Preparation**

# **Chunking**

In [4]:
PDF_FOLDER = Path(
    "/kaggle/input/datasets/shahddaymann/libraries-docs"
)

OUTPUT_FOLDER = Path(
    "/kaggle/working/final_chunks"
)

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

In [5]:
GUIDE_PDFS = {
    "NumPy_Complete_Guide.pdf",
    "pandas_complete_guide.pdf",
}



CHUNK_SIZE = 2500
CHUNK_OVERLAP = 300

# Remove chunks that contain almost no useful information
MIN_CHUNK_SIZE = 100


splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

In [6]:
def clean_text(text):
    """
    Clean common PDF extraction artifacts while preserving
    useful structure such as paragraphs and code.
    """

    if not text:
        return ""


    text = text.replace("\x7f", " ")
    text = text.replace("\u0000", " ")


    text = text.replace("●", "-")
    text = text.replace("•", "-")
    text = text.replace("▪", "-")
    text = text.replace("◦", "-")


    text = re.sub(
        r"(?m)^\s*l\s+",
        "- ",
        text
    )
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )


    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()


In [7]:
NUMBERED_HEADING = re.compile(
    r"^\s*(\d+(?:\.\d+)*)\.\s+(.+?)\s*$"
)
def detect_numbered_heading(line):
    """
    Detect headings such as:

        1. Introduction
        2. Installation
        2.1 Series
        2.2 DataFrame
        11.1 Merge
    """

    match = NUMBERED_HEADING.match(line)

    if not match:
        return None

    number = match.group(1)
    title = match.group(2).strip()

    return number, title

In [8]:
def is_noise(line):
    """
    Remove obvious PDF extraction artifacts.
    """

    line = line.strip()

    if not line:
        return True


    if re.fullmatch(
        r"Page\s+\d+",
        line,
        flags=re.IGNORECASE
    ):
        return True

    if re.fullmatch(
        r"\d+",
        line
    ):
        return True

    return False

In [9]:
def remove_duplicate_lines(text):
    """
    PDFs sometimes repeat headers/footers.

    Remove only consecutive duplicate lines so that
    legitimate repeated content is not destroyed.
    """

    lines = text.splitlines()

    cleaned_lines = []

    previous = None

    for line in lines:

        normalized = line.strip()

        if (
            normalized
            and normalized == previous
        ):
            continue

        cleaned_lines.append(line)

        previous = normalized

    return "\n".join(cleaned_lines)

In [10]:
#LOAD ALL PDFs

all_pages = []

pdf_files = sorted(
    PDF_FOLDER.glob("*.pdf")
)

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in: {PDF_FOLDER}"
    )


print("=" * 80)
print("LOADING PDF FILES")
print("=" * 80)


for pdf_path in pdf_files:

    print(
        f"Loading: {pdf_path.name}"
    )

    try:

        loader = PyPDFLoader(
            str(pdf_path)
        )

        pages = loader.load()

    except Exception as e:

        print(
            f"ERROR loading {pdf_path.name}: {e}"
        )

        continue

    library_name = pdf_path.stem

    source_type = (
        "guide"
        if pdf_path.name in GUIDE_PDFS
        else "reference"
    )

    for page in pages:

        page.metadata["library"] = (
            library_name
        )

        page.metadata["source"] = (
            pdf_path.name
        )

        page.metadata["source_type"] = (
            source_type
        )


        raw_page = page.metadata.get(
            "page",
            0
        )

        try:
            page_number = int(raw_page) + 1
        except:
            page_number = 1

        page.metadata["page"] = (
            page_number
        )

    all_pages.extend(pages)


print()
print("=" * 80)
print(
    f"TOTAL PDF PAGES LOADED: "
    f"{len(all_pages)}"
)
print("=" * 80)




pages_by_source = {}

for page in all_pages:

    source = page.metadata["source"]

    if source not in pages_by_source:
        pages_by_source[source] = []

    pages_by_source[source].append(page)



def process_guide(
    pages,
    library,
    source
):
    """
    Process small curated guide PDFs.

    A heading is kept together with its explanation/code.

    Example:

        4. Creating Arrays

        np.array(...)
        np.zeros(...)
        np.ones(...)

    becomes one semantic section rather than a heading-only
    chunk.
    """

    # --------------------------------------------------------
    # Combine pages
    # --------------------------------------------------------

    full_text = "\n\n".join(
        page.page_content
        for page in pages
    )

    full_text = clean_text(
        full_text
    )

    full_text = remove_duplicate_lines(
        full_text
    )

    lines = full_text.splitlines()

    sections = []

    current_section = None
    current_subsection = None
    current_content = []

    def save_current_section():

        nonlocal current_content

        if not current_content:
            return

        content = "\n".join(
            current_content
        ).strip()

        if not content:
            return

        sections.append(
            {
                "content": content,
                "section": current_section,
                "subsection": current_subsection,
            }
        )

        current_content = []

    # --------------------------------------------------------
    # Parse lines
    # --------------------------------------------------------

    for line in lines:

        line = line.strip()

        if is_noise(line):
            continue

        heading = detect_numbered_heading(
            line
        )

        # ====================================================
        # Numbered heading
        # ====================================================

        if heading:

            number, title = heading

            # ------------------------------------------------
            # Save previous content
            # ------------------------------------------------

            if current_content:
                save_current_section()

            # ------------------------------------------------
            # Main section
            #
            # 1. Introduction
            # 2. Installation
            # ------------------------------------------------

            if "." not in number:

                current_section = (
                    f"{number}. {title}"
                )

                current_subsection = None

            # ------------------------------------------------
            # Subsection
            #
            # 2.1 Series
            # 2.2 DataFrame
            # ------------------------------------------------

            else:

                current_subsection = (
                    f"{number}. {title}"
                )

            # ------------------------------------------------
            # IMPORTANT:
            # Heading is part of the content.
            # ------------------------------------------------

            current_content = [
                line
            ]

        else:

            current_content.append(
                line
            )

    # --------------------------------------------------------
    # Save final section
    # --------------------------------------------------------

    if current_content:
        save_current_section()

    # --------------------------------------------------------
    # Convert sections to Documents
    # --------------------------------------------------------

    final_documents = []

    chunk_counter = 0

    for section in sections:

        content = section[
            "content"
        ].strip()

        # ----------------------------------------------------
        # Ignore tiny sections
        # ----------------------------------------------------

        if len(content) < MIN_CHUNK_SIZE:
            continue

        # ----------------------------------------------------
        # Small enough → keep intact
        # ----------------------------------------------------

        if len(content) <= CHUNK_SIZE:

            final_documents.append(
                Document(
                    page_content=content,

                    metadata={
                        "library": library,
                        "source": source,
                        "source_type": "guide",

                        "section": section[
                            "section"
                        ],

                        "subsection": section[
                            "subsection"
                        ],

                        "page": None,

                        "chunk_id": (
                            f"{library}_"
                            f"guide_"
                            f"{chunk_counter}"
                        )
                    }
                )
            )

            chunk_counter += 1

        # ----------------------------------------------------
        # Too large → recursively split
        # ----------------------------------------------------

        else:

            sub_chunks = (
                splitter.split_text(
                    content
                )
            )

            for sub_chunk in sub_chunks:

                sub_chunk = (
                    sub_chunk.strip()
                )

                if (
                    len(sub_chunk)
                    < MIN_CHUNK_SIZE
                ):
                    continue

                final_documents.append(
                    Document(
                        page_content=sub_chunk,

                        metadata={
                            "library": library,
                            "source": source,
                            "source_type": "guide",

                            "section": section[
                                "section"
                            ],

                            "subsection": section[
                                "subsection"
                            ],

                            "page": None,

                            "chunk_id": (
                                f"{library}_"
                                f"guide_"
                                f"{chunk_counter}"
                            )
                        }
                    )
                )

                chunk_counter += 1

    return final_documents


# ============================================================
# 11. PROCESS REFERENCE PDFs
# ============================================================

def process_reference(
    pages,
    library,
    source
):
    """
    Process large reference PDFs.

    Each PDF page is cleaned and then recursively split.

    Page metadata is preserved for citations.
    """

    final_documents = []

    chunk_counter = 0

    for page in pages:

        # ----------------------------------------------------
        # Extract text
        # ----------------------------------------------------

        text = clean_text(
            page.page_content
        )

        if not text:
            continue

        # ----------------------------------------------------
        # Remove obvious noise lines
        # ----------------------------------------------------

        lines = []

        for line in text.splitlines():

            line = line.strip()

            if is_noise(line):
                continue

            lines.append(line)

        text = "\n".join(
            lines
        ).strip()

        if not text:
            continue

        # ----------------------------------------------------
        # Remove consecutive duplicate lines
        # ----------------------------------------------------

        text = remove_duplicate_lines(
            text
        )

        if len(text) < MIN_CHUNK_SIZE:
            continue

        # ----------------------------------------------------
        # Human-readable page number
        # ----------------------------------------------------

        page_number = page.metadata.get(
            "page",
            1
        )

        # ----------------------------------------------------
        # Split page
        # ----------------------------------------------------

        page_chunks = (
            splitter.split_text(
                text
            )
        )

        for chunk_text in page_chunks:

            chunk_text = (
                chunk_text.strip()
            )

            # ------------------------------------------------
            # Remove tiny chunks
            # ------------------------------------------------

            if (
                len(chunk_text)
                < MIN_CHUNK_SIZE
            ):
                continue

            # ------------------------------------------------
            # Create document
            # ------------------------------------------------

            final_documents.append(
                Document(
                    page_content=chunk_text,

                    metadata={
                        "library": library,
                        "source": source,
                        "source_type": "reference",

                        "page": page_number,

                        "section": None,
                        "subsection": None,

                        "chunk_id": (
                            f"{library}_"
                            f"reference_"
                            f"{chunk_counter}"
                        )
                    }
                )
            )

            chunk_counter += 1

    return final_documents


# ============================================================
# 12. CREATE FINAL CHUNKS
# ============================================================

all_chunks = []

print()
print("=" * 80)
print("CREATING CHUNKS")
print("=" * 80)


for source, pages in pages_by_source.items():

    library = pages[0].metadata[
        "library"
    ]

    source_type = pages[0].metadata[
        "source_type"
    ]

    print()
    print("=" * 80)
    print(
        f"Processing: {source}"
    )
    print(
        f"Type: {source_type}"
    )
    print(
        f"Pages: {len(pages)}"
    )
    print("=" * 80)

    # --------------------------------------------------------
    # Guide
    # --------------------------------------------------------

    if source_type == "guide":

        chunks = process_guide(
            pages=pages,
            library=library,
            source=source
        )

    # --------------------------------------------------------
    # Reference
    # --------------------------------------------------------

    else:

        chunks = process_reference(
            pages=pages,
            library=library,
            source=source
        )

    all_chunks.extend(chunks)

    print(
        f"Created {len(chunks)} chunks"
    )


# ============================================================
# 13. REMOVE EXACT DUPLICATE CHUNKS
# ============================================================

print()
print("=" * 80)
print("REMOVING DUPLICATE CHUNKS")
print("=" * 80)


unique_chunks = []

seen = set()

duplicates_removed = 0

for chunk in all_chunks:

    # Normalize text for duplicate detection
    normalized_text = re.sub(
        r"\s+",
        " ",
        chunk.page_content
    ).strip().lower()

    duplicate_key = (
        chunk.metadata["library"],
        normalized_text
    )

    if duplicate_key in seen:

        duplicates_removed += 1
        continue

    seen.add(
        duplicate_key
    )

    unique_chunks.append(
        chunk
    )


all_chunks = unique_chunks


print(
    f"Duplicates removed: "
    f"{duplicates_removed}"
)

print(
    f"Unique chunks: "
    f"{len(all_chunks)}"
)


# ============================================================
# 14. REASSIGN GLOBAL CHUNK IDs
# ============================================================

for global_id, chunk in enumerate(
    all_chunks
):

    chunk.metadata[
        "global_chunk_id"
    ] = global_id


# ============================================================
# 15. FINAL STATISTICS
# ============================================================

print()
print("=" * 80)
print("FINAL DATASET STATISTICS")
print("=" * 80)


print(
    f"Total final chunks: "
    f"{len(all_chunks)}"
)


# ------------------------------------------------------------
# Chunks by source
# ------------------------------------------------------------

source_counts = Counter(
    chunk.metadata["source"]
    for chunk in all_chunks
)


print()
print("Chunks by source:")

for source, count in (
    source_counts.items()
):

    print(
        f"{source}: {count}"
    )


# ------------------------------------------------------------
# Chunk sizes
# ------------------------------------------------------------

if all_chunks:

    sizes = [
        len(chunk.page_content)
        for chunk in all_chunks
    ]

    print()
    print("Chunk size statistics:")

    print(
        f"Minimum: "
        f"{min(sizes)} characters"
    )

    print(
        f"Maximum: "
        f"{max(sizes)} characters"
    )

    print(
        f"Average: "
        f"{sum(sizes) / len(sizes):.0f} characters"
    )

    print(
        f"Median: "
        f"{sorted(sizes)[len(sizes)//2]} characters"
    )


# ============================================================
# 16. GUIDE CHUNK PREVIEW
# ============================================================

print()
print("=" * 100)
print("GUIDE CHUNK PREVIEW")
print("=" * 100)


guide_chunks = [
    chunk
    for chunk in all_chunks
    if chunk.metadata[
        "source_type"
    ] == "guide"
]


for chunk in guide_chunks[:10]:

    print()
    print("=" * 100)

    print(
        "Chunk ID:",
        chunk.metadata[
            "chunk_id"
        ]
    )

    print(
        "Global ID:",
        chunk.metadata[
            "global_chunk_id"
        ]
    )

    print(
        "Library:",
        chunk.metadata[
            "library"
        ]
    )

    print(
        "Section:",
        chunk.metadata.get(
            "section"
        )
    )

    print(
        "Subsection:",
        chunk.metadata.get(
            "subsection"
        )
    )

    print(
        "Characters:",
        len(
            chunk.page_content
        )
    )

    print()
    print("CONTENT:")
    print(
        chunk.page_content[:2000]
    )


# ============================================================
# 17. REFERENCE CHUNK PREVIEW
# ============================================================

print()
print("=" * 100)
print("REFERENCE CHUNK PREVIEW")
print("=" * 100)


reference_chunks = [
    chunk
    for chunk in all_chunks
    if chunk.metadata[
        "source_type"
    ] == "reference"
]


for chunk in reference_chunks[:5]:

    print()
    print("=" * 100)

    print(
        "Chunk ID:",
        chunk.metadata[
            "chunk_id"
        ]
    )

    print(
        "Global ID:",
        chunk.metadata[
            "global_chunk_id"
        ]
    )

    print(
        "Library:",
        chunk.metadata[
            "library"
        ]
    )

    print(
        "Source:",
        chunk.metadata[
            "source"
        ]
    )

    print(
        "Page:",
        chunk.metadata[
            "page"
        ]
    )

    print(
        "Characters:",
        len(
            chunk.page_content
        )
    )

    print()
    print("CONTENT:")
    print(
        chunk.page_content[:1500]
    )


# ============================================================
# 18. SAVE MARKDOWN FILES
# ============================================================

print()
print("=" * 80)
print("SAVING MARKDOWN FILES")
print("=" * 80)


for source in pages_by_source.keys():

    source_chunks = [
        chunk
        for chunk in all_chunks
        if chunk.metadata[
            "source"
        ] == source
    ]

    if not source_chunks:
        continue

    library = source_chunks[0].metadata[
        "library"
    ]

    output_file = (
        OUTPUT_FOLDER
        / f"{Path(source).stem}_chunks.md"
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            f"# {library}\n\n"
        )

        f.write(
            f"Source: `{source}`\n\n"
        )

        f.write(
            f"Total chunks: "
            f"{len(source_chunks)}\n\n"
        )

        for chunk in source_chunks:

            f.write(
                f"## Chunk: "
                f"{chunk.metadata['chunk_id']}\n\n"
            )

            f.write(
                f"**Global ID:** "
                f"{chunk.metadata['global_chunk_id']}\n\n"
            )

            f.write(
                f"**Library:** "
                f"{chunk.metadata['library']}\n\n"
            )

            f.write(
                f"**Source:** "
                f"{chunk.metadata['source']}\n\n"
            )

            f.write(
                f"**Source Type:** "
                f"{chunk.metadata['source_type']}\n\n"
            )

            if chunk.metadata.get(
                "section"
            ):

                f.write(
                    f"**Section:** "
                    f"{chunk.metadata['section']}\n\n"
                )

            if chunk.metadata.get(
                "subsection"
            ):

                f.write(
                    f"**Subsection:** "
                    f"{chunk.metadata['subsection']}\n\n"
                )

            if chunk.metadata.get(
                "page"
            ) is not None:

                f.write(
                    f"**Page:** "
                    f"{chunk.metadata['page']}\n\n"
                )

            f.write(
                "### Content\n\n"
            )

            f.write(
                chunk.page_content
            )

            f.write(
                "\n\n---\n\n"
            )

    print(
        f"Saved: {output_file}"
    )


# ============================================================
# 19. SAVE JSONL
# ============================================================

jsonl_file = (
    OUTPUT_FOLDER
    / "all_chunks.jsonl"
)


with open(
    jsonl_file,
    "w",
    encoding="utf-8"
) as f:

    for chunk in all_chunks:

        record = {
            "chunk_id": chunk.metadata[
                "chunk_id"
            ],

            "global_chunk_id": chunk.metadata[
                "global_chunk_id"
            ],

            "text": chunk.page_content,

            "metadata": chunk.metadata
        }

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )


print(
    f"Saved: {jsonl_file}"
)


# ============================================================
# 20. FINAL CHECK
# ============================================================

print()
print("=" * 80)
print("FINAL CHECK")
print("=" * 80)


tiny_chunks = [
    chunk
    for chunk in all_chunks
    if len(chunk.page_content)
    < MIN_CHUNK_SIZE
]


print(
    f"Chunks below {MIN_CHUNK_SIZE} chars: "
    f"{len(tiny_chunks)}"
)


if tiny_chunks:

    print(
        "WARNING: Tiny chunks still exist."
    )

else:

    print(
        "✓ No tiny chunks."
    )


print(
    f"Total chunks ready for embeddings: "
    f"{len(all_chunks)}"
)

print()
print(
    "Chunking stage completed successfully."
)

LOADING PDF FILES
Loading: NumPy_Complete_Guide.pdf
Loading: numpy-ref.pdf
Loading: pandas_complete_guide.pdf
Loading: scikit-learn-docs.pdf


Multiple definitions in dictionary at byte 0x2c4f47e for key /Author
Multiple definitions in dictionary at byte 0x2c4f486 for key /Title



TOTAL PDF PAGES LOADED: 4184

CREATING CHUNKS

Processing: NumPy_Complete_Guide.pdf
Type: guide
Pages: 12
Created 23 chunks

Processing: numpy-ref.pdf
Type: reference
Pages: 2101
Created 2265 chunks

Processing: pandas_complete_guide.pdf
Type: guide
Pages: 11
Created 19 chunks

Processing: scikit-learn-docs.pdf
Type: reference
Pages: 2060
Created 2299 chunks

REMOVING DUPLICATE CHUNKS
Duplicates removed: 0
Unique chunks: 4606

FINAL DATASET STATISTICS
Total final chunks: 4606

Chunks by source:
NumPy_Complete_Guide.pdf: 23
numpy-ref.pdf: 2265
pandas_complete_guide.pdf: 19
scikit-learn-docs.pdf: 2299

Chunk size statistics:
Minimum: 126 characters
Maximum: 2500 characters
Average: 1618 characters
Median: 1678 characters

GUIDE CHUNK PREVIEW

Chunk ID: NumPy_Complete_Guide_guide_0
Global ID: 0
Library: NumPy_Complete_Guide
Section: None
Subsection: None
Characters: 299

CONTENT:
NumPy — The Complete Practical Guide
NumPy
The Complete Practical Guide
Arrays Indexing Broadcasting Math & L

In [11]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.2 MB/s eta 0:00:0000:01:00:01


In [12]:
import pickle
import numpy as np
import torch
import faiss

from sentence_transformers import SentenceTransformer


CHUNKS_FILE = Path("/kaggle/working/final_chunks/all_chunks.jsonl")

VECTOR_FOLDER = Path("/kaggle/working/vector_store")

VECTOR_FOLDER.mkdir(
    parents=True,
    exist_ok=True)


if not CHUNKS_FILE.exists():

    raise FileNotFoundError(
        f"Chunks file not found:\n{CHUNKS_FILE}"
    )


print("=" * 80)
print("PYTHON DOCUMENTATION RAG - EMBEDDING STAGE")
print("=" * 80)
print(
    f"Chunks file: {CHUNKS_FILE}"
)

PYTHON DOCUMENTATION RAG - EMBEDDING STAGE
Chunks file: /kaggle/working/final_chunks/all_chunks.jsonl


In [13]:
chunks = []

with open(
    CHUNKS_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        record = json.loads(line)

        chunks.append(record)


print()
print(f"Loaded chunks: {len(chunks)}")

if not chunks:
    raise ValueError("No chunks were loaded.")

print()
print("=" * 80)
print("EXAMPLE CHUNK")
print("=" * 80)

example = chunks[0]

print(
    "Chunk ID:",
    example["chunk_id"])

print("Library:",example["metadata"].get("library"))

print("Source:",example["metadata"].get("source"))

print("Page:",example["metadata"].get("page"))

print()
print(example["text"][:1000])


Loaded chunks: 4606

EXAMPLE CHUNK
Chunk ID: NumPy_Complete_Guide_guide_0
Library: NumPy_Complete_Guide
Source: NumPy_Complete_Guide.pdf
Page: None

NumPy — The Complete Practical Guide
NumPy
The Complete Practical Guide
Arrays Indexing Broadcasting Math & Linear Algebra Statistics Random Sampling I/O
Performance
A reference document with runnable code examples for every core NumPy feature.
NumPy — The Complete Practical Guide
Table of Contents


In [14]:
if torch.cuda.is_available():

    device = "cuda"

    print()
    print(
        "GPU detected:"
    )

    print(
        torch.cuda.get_device_name(0)
    )

else:

    device = "cpu"
    print()
    print("No GPU detected.")

    print("Using CPU.")


MODEL_NAME = (
    "BAAI/bge-small-en-v1.5"
)


print()
print("=" * 80)

print("LOADING EMBEDDING MODEL")
print("=" * 80)

print(f"Model: {MODEL_NAME}")

print(f"Device: {device}")


model = SentenceTransformer(
    MODEL_NAME,
    device=device
)

texts = [
    chunk["text"]
    for chunk in chunks
]


print()
print(f"Texts to embed: {len(texts)}")


GPU detected:
Tesla T4

LOADING EMBEDDING MODEL
Model: BAAI/bge-small-en-v1.5
Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Texts to embed: 4606


In [15]:
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    device=device
)

print()

print("EMBEDDING INFORMATION")
print("=" * 80)

print("Shape:",embeddings.shape)

print("Data type:",embeddings.dtype)

print("Expected number of vectors:",len(chunks))

print("Embedding dimension:",embeddings.shape[1])


Batches:   0%|          | 0/144 [00:00<?, ?it/s]


EMBEDDING INFORMATION
Shape: (4606, 384)
Data type: float32
Expected number of vectors: 4606
Embedding dimension: 384


In [16]:
embeddings = np.asarray(
    embeddings,
    dtype="float32"
)

print()
print("BUILDING FAISS INDEX")
print("=" * 80)

embedding_dimension = (embeddings.shape[1])


BUILDING FAISS INDEX


In [17]:
index = faiss.IndexFlatIP(embedding_dimension)
index.add(embeddings)

print(
    f"FAISS vectors stored: "
    f"{index.ntotal}"
)

print(
    f"Vector dimension: "
    f"{embedding_dimension}")


INDEX_FILE = (
    VECTOR_FOLDER
    / "python_docs.faiss")


faiss.write_index(index,str(INDEX_FILE))


print()
print(f"FAISS index saved to:")
print(INDEX_FILE)

FAISS vectors stored: 4606
Vector dimension: 384

FAISS index saved to:
/kaggle/working/vector_store/python_docs.faiss


In [18]:
METADATA_FILE = (
    VECTOR_FOLDER
    / "chunk_metadata.pkl")

with open(
    METADATA_FILE,
    "wb"
) as f:
    pickle.dump(chunks,f)


print()
print("Chunk metadata saved to:")
print(METADATA_FILE)

EMBEDDINGS_FILE = (
    VECTOR_FOLDER
    / "embeddings.npy")

np.save(
    EMBEDDINGS_FILE,
    embeddings
)


print()
print("Embeddings saved to:")
print(EMBEDDINGS_FILE)


Chunk metadata saved to:
/kaggle/working/vector_store/chunk_metadata.pkl

Embeddings saved to:
/kaggle/working/vector_store/embeddings.npy


In [19]:
print("Summary")
print()
print(f"Total chunks:       {len(chunks)}")

print(f"Embedding dimension: {embedding_dimension}")

print(f"FAISS vectors:       {index.ntotal}")

print(f"Model:               {MODEL_NAME}")

print(f"Device:              {device}")

print()
print("Files created:")

print(f"1. {INDEX_FILE}")

print(f"2. {METADATA_FILE}")

print(f"3. {EMBEDDINGS_FILE}")

Summary

Total chunks:       4606
Embedding dimension: 384
FAISS vectors:       4606
Model:               BAAI/bge-small-en-v1.5
Device:              cuda

Files created:
1. /kaggle/working/vector_store/python_docs.faiss
2. /kaggle/working/vector_store/chunk_metadata.pkl
3. /kaggle/working/vector_store/embeddings.npy


In [20]:
import faiss
import pickle
import numpy as np

FAISS_PATH = (
    "/kaggle/working/vector_store/"
    "python_docs.faiss"
)

METADATA_PATH = (
    "/kaggle/working/vector_store/"
    "chunk_metadata.pkl"
)

EMBEDDINGS_PATH = (
    "/kaggle/working/vector_store/"
    "embeddings.npy"
)

index = faiss.read_index(
    FAISS_PATH
)

with open(
    METADATA_PATH,
    "rb"
) as f:

    chunk_metadata = pickle.load(f)

embeddings = np.load(
    EMBEDDINGS_PATH
)

print("FAISS vectors:", index.ntotal)
print("Metadata:", len(chunk_metadata))
print("Embeddings:", embeddings.shape)

FAISS vectors: 4606
Metadata: 4606
Embeddings: (4606, 384)


# **Retrieval**

In [21]:
from sentence_transformers import SentenceTransformer
import torch

EMBEDDING_MODEL_NAME = (
    "BAAI/bge-small-en-v1.5"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [22]:
def faiss_search(
    query,
    top_k=5
):
    # 1. Create query embedding
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )
    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    # 2. Search FAISS

    scores, indices = index.search(
        query_embedding,
        top_k
    )
    # 3. Build results

    results = []


    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        # Skip invalid FAISS indices
        if idx < 0:
            continue

        metadata = chunk_metadata[idx]

        # Extract text

        text = metadata.get(
            "text",
            ""
        )
        # Store result

        results.append({

            "rank": rank,
            "index": int(idx),
            "score": float(score),
            "text": text,
            "metadata": metadata

        })
    return results

In [23]:
query = "How do I create a NumPy array?"

results = faiss_search(
    query,
    top_k=10
)

print("QUERY:")
print(query)

for result in results:

    print("\n" + "=" * 80)

    print("Rank:", result["rank"])
    print(
        "FAISS score:",
        round(result["score"], 4)
    )

    print(
        "Library:",
        result["metadata"].get("library")
    )

    print("\nCONTENT:")
    print(
        result["text"][:800]
    )

QUERY:
How do I create a NumPy array?

Rank: 1
FAISS score: 0.8252
Library: None

CONTENT:
4. Creating Arrays
4.1 From Python Lists / Tuples
a = np.array([1, 2, 3]) # 1-D array
b = np.array([[1, 2, 3], [4, 5, 6]]) # 2-D array (2 rows, 3 cols)
c = np.array([1, 2, 3], dtype=np.float32) # specify dtype explicitly
4.2 Built-in Array Constructors
np.zeros((2, 3)) # 2x3 array of zeros
np.ones((3, 3)) # 3x3 array of ones
np.full((2, 2), 7) # 2x2 array filled with the value 7
np.empty((2, 3)) # uninitialized array (fast, garbage values)
np.eye(3) # 3x3 identity matrix
np.identity(4) # 4x4 identity matrix
np.diag([1, 2, 3]) # diagonal matrix from a 1-D array
4.3 Ranges and Sequences
np.arange(0, 10, 2) # array([0, 2, 4, 6, 8]) (like range())
np.linspace(0, 1, 5) # 5 evenly spaced values between 0 and 1
np.logspace(0, 3, 4) # array([1, 10, 100, 1000]) log-spaced values
4.4 Random Arrays
n

Rank: 2
FAISS score: 0.784
Library: None

CONTENT:
NumPyReference,Release2.5.0
Notes
Thisconstructorcanbeco

In [27]:
!pip install -q sentence-transformers

In [24]:
from sentence_transformers import CrossEncoder

RERANKER_NAME = (
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

reranker = CrossEncoder(
    RERANKER_NAME,
    device=(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )
)

print("Reranker loaded")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded


In [25]:
def rerank_results(
    query,
    results,
    top_k=5
):

    pairs = [
        (
            query,
            result["text"]
        )
        for result in results
    ]

    scores = reranker.predict(
        pairs
    )

    for result, score in zip(
        results,
        scores
    ):

        result["rerank_score"] = float(
            score
        )

    results = sorted(
        results,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    for rank, result in enumerate(
        results[:top_k],
        start=1
    ):

        result["rerank_rank"] = rank

    return results[:top_k]

In [26]:
query = "How do I create a NumPy array?"

candidates = faiss_search(
    query,
    top_k=20
)

final_results = rerank_results(
    query,
    candidates,
    top_k=5
)

for result in final_results:

    print("\n" + "=" * 80)

    print(
        "Rerank:",
        result["rerank_rank"]
    )

    print(
        "FAISS score:",
        round(result["score"], 4)
    )

    print(
        "Reranker score:",
        round(
            result["rerank_score"],
            4
        )
    )

    print(
        "Library:",
        result["metadata"].get(
            "library"
        )
    )

    print("\n", result["text"][:1000])


Rerank: 1
FAISS score: 0.7763
Reranker score: 4.6015
Library: None

 NumPyReference,Release2.5.0
Notes
This function is the preferred method for creating an array copy. The functionnumpy.copy is similar, but it
defaultstousingorder‘K’,andwillnotpasssub-classesthroughbydefault.
Examples
>>> import numpy as np
>>> x = np.array([[1,2,3],[4,5,6]], order ='F')
>>> y = x.copy()
>>> x.fill(0)
>>> x
array([[0, 0, 0],
[0, 0, 0]])
>>> y
array([[1, 2, 3],
[4, 5, 6]])
>>> y.flags['C_CONTIGUOUS']
True
ForarrayscontainingPythonobjects(e.g. dtype=np.object_),thecopyisashallowone. Thenewarraywillcontain
thesameobjectwhichmayleadtosurprisesifthatobjectcanbemodified(ismutable):
>>> a = np.array([1, 'm', [ 2, 3, 4]], dtype =np.object_)
>>> b = a.copy()
>>> b[2][0] = 10
>>> a
array([1, 'm', list([10, 3, 4])], dtype=object)
Toensureallelementswithinan objectarrayarecopied,use copy.deepcopy:
>>> import copy
>>> a = np.array([1, 'm', [ 2, 3, 4]], dtype =np.object_)
>>> c = copy.deepcopy(a)
>>> c[2][0] = 10


In [27]:
!pip install -q -U google-genai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 828.7 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 1.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 9.3 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 29.3 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.57.0 which is incompatible.
google-colab 1.0.0 re

In [29]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 1.2 MB/s eta 0:00:00a 0:00:01


In [30]:
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv(
    "/kaggle/working/.env",
    override=True
)

API_KEY = os.getenv("GROQ_API_KEY")

ROUTER_MODEL = os.getenv("GROQ_MODEL")
GENERATION_MODEL = os.getenv("GROQ_MODEL")

print("API key loaded:", bool(API_KEY))
print("Router:", ROUTER_MODEL)
print("Generator:", GENERATION_MODEL)

client = Groq(
    api_key=API_KEY
)

print("Groq client ready")

API key loaded: True
Router: openai/gpt-oss-20b
Generator: openai/gpt-oss-20b
Groq client ready


In [59]:
#response = client.chat.completions.create(
#    model=GENERATION_MODEL,
#    messages=[
#        {
#            "role": "user",
#            "content": "What is a NumPy array? Answer in one sentence."
#        }
#    ]
#)
#
#print(response.choices[0].message.content)

A NumPy array is a homogeneous, multi‑dimensional container that stores data in a contiguous block of memory for fast numerical operations.


In [31]:
# QUERY ROUTER

def route_query(query):

    prompt = f"""
You are a query router for a Python documentation assistant.

The system has a RAG knowledge base containing documentation
for exactly these libraries:

- NumPy
- Pandas
- Scikit-learn

Classify the user's question into EXACTLY ONE of these
three categories:

============================================================
1. RETRIEVE
============================================================

Choose RETRIEVE when the question is specifically about:

- NumPy
- Pandas
- Scikit-learn
- Their functions
- Their classes
- Their APIs
- Their usage
- Their parameters
- Their behavior
- Their examples
- Their errors
- Their documentation

Examples:

"How do I create a NumPy array?"
RETRIEVE

"How does pandas merge work?"
RETRIEVE

"What parameters does sklearn train_test_split accept?"
RETRIEVE

"How can I normalize data using scikit-learn?"
RETRIEVE


============================================================
2. GENERAL
============================================================

Choose GENERAL when the question is about Python itself
or general programming concepts, but is NOT specifically
about NumPy, Pandas, or Scikit-learn.

Examples:

"What is a Python list?"
GENERAL

"What is a tuple in Python?"
GENERAL

"How do Python decorators work?"
GENERAL

"What is a Python dictionary?"
GENERAL

"How does a for loop work in Python?"
GENERAL

"What is object-oriented programming?"
GENERAL


============================================================
3. OUT_OF_CONTEXT
============================================================

Choose OUT_OF_CONTEXT when the question is unrelated to:

- Python
- Programming
- NumPy
- Pandas
- Scikit-learn

Examples:

"What is the capital of France?"
OUT_OF_CONTEXT

"Who won the football match?"
OUT_OF_CONTEXT

"Tell me a joke."
OUT_OF_CONTEXT

"What is the weather today?"
OUT_OF_CONTEXT

"How do I cook pasta?"
OUT_OF_CONTEXT


============================================================
IMPORTANT RULES
============================================================

A question about NumPy, Pandas, or Scikit-learn MUST be
RETRIEVE even if it is also a Python question.

A general Python question that does not require the
NumPy/Pandas/Scikit-learn documentation is GENERAL.

A completely unrelated question is OUT_OF_CONTEXT.

Return ONLY ONE of:

RETRIEVE
GENERAL
OUT_OF_CONTEXT

Do not return explanations.
Do not return punctuation.
Do not return anything else.

User question:
{query}
"""

    response = client.chat.completions.create(
        model=ROUTER_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    route = response.choices[0].message.content.strip().upper()

    valid_routes = {
        "RETRIEVE",
        "GENERAL",
        "OUT_OF_CONTEXT"
    }

    if route not in valid_routes:
        route = "OUT_OF_CONTEXT"

    return route

In [32]:
#test_queries = [
#
#    "How do I create a NumPy array?",
#
#    "How do I merge two pandas DataFrames?",
#
#    "What is train_test_split in scikit-learn?",
#
#    "What is a Python list?",
#
#    "How do Python decorators work?",
#
#    "What is a dictionary in Python?",
#
#    "What is the capital of France?",
#
#    "Tell me a joke."
#]
#
#for query in test_queries:
#
#    route = route_query(query)
#
#    print("=" * 80)
#    print("Query:", query)
#    print("Route:", route)

Query: How do I create a NumPy array?
Route: RETRIEVE


In [33]:
PYTHON_DOCS_URL = "https://docs.python.org/3/"


def general_answer(query):

    return (
        "For general Python documentation, visit the official "
        "Python documentation:\n"
        f"{PYTHON_DOCS_URL}"
    )

In [34]:
def out_of_context_answer():

    return "This question is out of context."

In [35]:
def build_context(results):

    context_parts = []

    for i, result in enumerate(
        results,
        start=1
    ):

        metadata = result["metadata"]

        library = metadata.get(
            "library",
            "Unknown"
        )

        source = metadata.get(
            "source",
            "Unknown"
        )

        page = metadata.get(
            "page",
            None
        )

        header = (
            f"[Document {i}]\n"
            f"Library: {library}\n"
            f"Source: {source}\n"
        )

        if page is not None:

            header += (
                f"Page: {page}\n"
            )

        context_parts.append(
            header
            + "\n"
            + result["text"]
        )

    return "\n\n".join(
        context_parts
    )

In [36]:
def rag_answer(query, context):

    prompt = f"""
You are a Python documentation assistant.

Your knowledge source for this answer is ONLY the
documentation provided below.

The documentation comes from:

- NumPy
- Pandas
- Scikit-learn

Answer the user's question using ONLY the provided context.

Do NOT use outside knowledge.

Do NOT invent information.

If the provided context does not contain enough information
to answer the question, say:

"I don't know based on the provided documentation."

Keep the answer clear and concise.

============================================================
DOCUMENTATION CONTEXT
============================================================

{context}

============================================================
USER QUESTION
============================================================

{query}

============================================================
ANSWER
============================================================
"""

    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [37]:
def answer_query(query):
    # 1. ROUTE THE QUERY

    route = route_query(query)

    print("Route:", route)

    # 2. RETRIEVE
    if route == "RETRIEVE":

        # FAISS search
        results = faiss_search(
            query,
            top_k=5
        )
        # Build context

        context_parts = []

        for result in results:

            text = result.get("text", "")

            if text:
                context_parts.append(text)

        context = "\n\n---\n\n".join(
            context_parts
        )

        # Generate answer

        answer = rag_answer(
            query,
            context
        )

        return {
            "route": "RETRIEVE",
            "answer": answer,
            "results": results
        }

    # 3. GENERAL PYTHON

    elif route == "GENERAL":

        answer = general_answer(query)

        return {
            "route": "GENERAL",
            "answer": answer,
            "results": []
        }

    # 4. OUT OF CONTEXT
    else:

        answer = out_of_context_answer()

        return {
            "route": "OUT_OF_CONTEXT",
            "answer": answer,
            "results": []
        }

In [38]:
result = answer_query(
    "what does standard scaler do?"
)

print(result["route"])
print()
print(result["answer"])

Route: RETRIEVE
RETRIEVE

**StandardScaler** removes the mean and scales each feature to unit variance.  
It computes the mean and standard deviation on a training set and then transforms data by

\[
X_{\text{scaled}} = \frac{X - \text{mean}}{\text{std}}
\]

so that the transformed features have zero mean and unit variance. This is useful for many estimators that assume centered data with comparable scales.


In [43]:
#import shutil
#
#shutil.make_archive(
#    "/kaggle/working/vector_store",
#    "zip",
#    "/kaggle/working/vector_store"
#)
#
#print("Created:")
#print("/kaggle/working/vector_store.zip")

In [39]:
evaluation_questions = [
    {
        "question": "How do I create a NumPy array?",
        "expected_route": "RETRIEVE"
    },
    {
        "question": "How do I reshape a NumPy array?",
        "expected_route": "RETRIEVE"
    },
    {
        "question": "How do I concatenate arrays in NumPy?",
        "expected_route": "RETRIEVE"
    },
    {
        "question": "How do I create a DataFrame in pandas?",
        "expected_route": "RETRIEVE"
    },
    {
        "question": "What does train_test_split do in scikit-learn?",
        "expected_route": "RETRIEVE"
    },
    {
        "question": "How do I normalize data using scikit-learn?",
        "expected_route": "RETRIEVE"
    },
    {
        "question": "What is a Python list?",
        "expected_route": "GENERAL"
    },
    {
        "question": "What is the capital of France?",
        "expected_route": "OUT_OF_CONTEXT"
    },
    {   "question": "What is a python dictionary?",
        "expected_route": "GENERAL"},
    {   "question": "where is alexandria located?",
        "expected_route": "OUT_OF_CONTEXT"}
]

print(
    f"Number of evaluation questions: "
    f"{len(evaluation_questions)}"
)

Number of evaluation questions: 10


In [40]:
def evaluate_question(question):
    route = route_query(question)
    if route == "OUT_OF_CONTEXT":

        answer = "This question is out of context."

        return {
            "question": question,
            "route": route,
            "retrieved_source": "N/A",
            "retrieved_context": "",
            "answer": answer
        }

    if route == "GENERAL":

        answer = general_answer(question)

        return {
            "question": question,
            "route": route,
            "retrieved_source": "Official Python documentation",
            "retrieved_context": "",
            "answer": answer
        }
    results = faiss_search(
        question,
        top_k=5
    )

    context_parts = []

    for result in results:

        context_parts.append(
            result["text"]
        )

    context = "\n\n".join(
        context_parts
    )

    answer = rag_answer(
        question,
        context
    )
    sources = []

    for result in results:

        metadata = result.get(
            "metadata",
            {}
        )

        sources.append(
            {
                "rank": result.get("rank"),
                "score": result.get("score"),
                "library": metadata.get("library"),
                "source": metadata.get("source"),
                "page": metadata.get("page")
            }
        )


    return {
        "question": question,
        "route": route,
        "retrieved_source": sources,
        "retrieved_context": context,
        "answer": answer
    }

In [41]:
evaluation_results = []

for item in evaluation_questions:

    question = item["question"]

    print("=" * 80)
    print("Question:", question)

    result = evaluate_question(
        question
    )

    evaluation_results.append(
        result
    )

    print("Route:", result["route"])
    print("Answer:", result["answer"])

    print()

Question: How do I create a NumPy array?
Route: RETRIEVE
Answer: You can create a NumPy array in several ways:

| Method | Code example | What it does |
|--------|--------------|--------------|
| **From a Python list or tuple** | `a = np.array([1, 2, 3])`<br>`b = np.array([[1, 2, 3], [4, 5, 6]])` | Builds an array with the same shape as the nested list. |
| **Specify the data type** | `c = np.array([1, 2, 3], dtype=np.float32)` | Creates the array with the given dtype. |
| **Built‑in constructors** | `np.zeros((2, 3))`<br>`np.ones((3, 3))`<br>`np.full((2, 2), 7)`<br>`np.empty((2, 3))`<br>`np.eye(3)`<br>`np.identity(4)`<br>`np.diag([1, 2, 3])` | Generates arrays of zeros, ones, a constant value, uninitialized values, identity matrices, or a diagonal matrix. |
| **Ranges and sequences** | `np.arange(0, 10, 2)`<br>`np.linspace(0, 1, 5)`<br>`np.logspace(0, 3, 4)` | Creates evenly spaced values or log‑spaced values. |
| **Random arrays** | `np.random.rand(2, 3)`<br>`np.random.randn(2, 3)`<b

In [42]:
import pandas as pd


evaluation_table = []

for result in evaluation_results:

    sources = result["retrieved_source"]

    if isinstance(sources, list):

        source_text = "; ".join(
            [
                (
                    f"{s.get('library')} | "
                    f"{s.get('source')} | "
                    f"Page {s.get('page')}"
                )
                for s in sources
            ]
        )

    else:

        source_text = sources


    evaluation_table.append(
        {
            "Question": result["question"],
            "Route": result["route"],
            "Retrieved Source": source_text,
            "Answer": result["answer"],

            "Context Relevant": "",
            "Grounded": "",
            "Correct": ""
        }
    )


evaluation_df = pd.DataFrame(
    evaluation_table
)


display(
    evaluation_df
)

,Question,Route,Retrieved Source,Answer,Context Relevant,Grounded,Correct
0,How do I create a NumPy array?,RETRIEVE,None | None | Page None; None | None | Page No...,You can create a NumPy array in several ways:\...,,,
1,How do I reshape a NumPy array?,RETRIEVE,None | None | Page None; None | None | Page No...,To change the shape of a NumPy array you use *...,,,
2,How do I concatenate arrays in NumPy?,RETRIEVE,None | None | Page None; None | None | Page No...,To concatenate arrays in NumPy you use **`np.c...,,,
3,How do I create a DataFrame in pandas?,RETRIEVE,None | None | Page None; None | None | Page No...,"To create a DataFrame in pandas, use the `pd.D...",,,
4,What does train_test_split do in scikit-learn?,RETRIEVE,None | None | Page None; None | None | Page No...,`train_test_split` is a convenience function t...,,,
5,How do I normalize data using scikit-learn?,RETRIEVE,None | None | Page None; None | None | Page No...,**Normalizing data in scikit‑learn**\n\nscikit...,,,
6,What is a Python list?,GENERAL,Official Python documentation,"For general Python documentation, visit the of...",,,
7,What is the capital of France?,OUT_OF_CONTEXT,N/A,This question is out of context.,,,
8,What is a python dictionary?,GENERAL,Official Python documentation,"For general Python documentation, visit the of...",,,
9,where is alexandria located?,OUT_OF_CONTEXT,N/A,This question is out of context.,,,


In [43]:
evaluation_judgments = {
    "How do I create a NumPy array?": {
        "Context Relevant": "Yes",
        "Grounded": "Yes",
        "Correct": "Yes"
    },

    "How do I reshape a NumPy array?": {
        "Context Relevant": "Yes",
        "Grounded": "Yes",
        "Correct": "Yes"
    },

    "How do I concatenate arrays in NumPy?": {
        "Context Relevant": "Yes",
        "Grounded": "Yes",
        "Correct": "Yes"
    },

    "How do I create a DataFrame in pandas?": {
        "Context Relevant": "Yes",
        "Grounded": "Yes",
        "Correct": "Yes"
    },

    "What does train_test_split do in scikit-learn?": {
        "Context Relevant": "Yes",
        "Grounded": "Yes",
        "Correct": "Yes"
    },

    "How do I normalize data using scikit-learn?": {
        "Context Relevant": "Yes",
        "Grounded": "Yes",
        "Correct": "Yes"
    },

    "What is a Python list?": {
        "Context Relevant": "N/A",
        "Grounded": "N/A",
        "Correct": "Yes"
    },

    "What is the capital of France?": {
        "Context Relevant": "N/A",
        "Grounded": "N/A",
        "Correct": "Yes"
    },

    "What is a python dictionary?": {
        "Context Relevant": "N/A",
        "Grounded": "N/A",
        "Correct": "Yes"
    },

    "where is alexandria located?": {
        "Context Relevant": "N/A",
        "Grounded": "N/A",
        "Correct": "Yes"
    }
}
for question, judgment in evaluation_judgments.items():

    mask = evaluation_df["Question"] == question

    for column, value in judgment.items():

        evaluation_df.loc[
            mask,
            column
        ] = value

display(evaluation_df)

,Question,Route,Retrieved Source,Answer,Context Relevant,Grounded,Correct
0,How do I create a NumPy array?,RETRIEVE,None | None | Page None; None | None | Page No...,You can create a NumPy array in several ways:\...,Yes,Yes,Yes
1,How do I reshape a NumPy array?,RETRIEVE,None | None | Page None; None | None | Page No...,To change the shape of a NumPy array you use *...,Yes,Yes,Yes
2,How do I concatenate arrays in NumPy?,RETRIEVE,None | None | Page None; None | None | Page No...,To concatenate arrays in NumPy you use **`np.c...,Yes,Yes,Yes
3,How do I create a DataFrame in pandas?,RETRIEVE,None | None | Page None; None | None | Page No...,"To create a DataFrame in pandas, use the `pd.D...",Yes,Yes,Yes
4,What does train_test_split do in scikit-learn?,RETRIEVE,None | None | Page None; None | None | Page No...,`train_test_split` is a convenience function t...,Yes,Yes,Yes
5,How do I normalize data using scikit-learn?,RETRIEVE,None | None | Page None; None | None | Page No...,**Normalizing data in scikit‑learn**\n\nscikit...,Yes,Yes,Yes
6,What is a Python list?,GENERAL,Official Python documentation,"For general Python documentation, visit the of...",N/A,N/A,Yes
7,What is the capital of France?,OUT_OF_CONTEXT,N/A,This question is out of context.,N/A,N/A,Yes
8,What is a python dictionary?,GENERAL,Official Python documentation,"For general Python documentation, visit the of...",N/A,N/A,Yes
9,where is alexandria located?,OUT_OF_CONTEXT,N/A,This question is out of context.,N/A,N/A,Yes


In [44]:
valid_df = evaluation_df[
    evaluation_df["Route"] == "RETRIEVE"
].copy()


def percentage_yes(column):

    values = valid_df[column].astype(str).str.lower()

    if len(values) == 0:
        return 0.0

    return (
        (values == "yes").sum()
        / len(values)
        * 100
    )


retrieval_relevance = percentage_yes(
    "Context Relevant"
)

grounded_percentage = percentage_yes(
    "Grounded"
)

correct_percentage = percentage_yes(
    "Correct"
)


print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)

print(
    f"Relevant retrieval: "
    f"{retrieval_relevance:.1f}%"
)

print(
    f"Grounded answers: "
    f"{grounded_percentage:.1f}%"
)

print(
    f"Correct answers: "
    f"{correct_percentage:.1f}%"
)

EVALUATION RESULTS
Relevant retrieval: 100.0%
Grounded answers: 100.0%
Correct answers: 100.0%


In [45]:
evaluation_path = (
    "/kaggle/working/evaluation_results.csv"
)

evaluation_df.to_csv(
    evaluation_path,
    index=False
)

print(
    "Saved evaluation results to:"
)

print(
    evaluation_path
)

Saved evaluation results to:
/kaggle/working/evaluation_results.csv
